# Day 16: Week 3 综合复习 —— 知识速查 + 综合场景演练

> **目标**: 回顾 Week 3 三天核心内容，通过综合场景串联所有 Pandas 知识点。
> **建议用时**: 30 分钟回顾 + 60 分钟习题

## 1. Pandas 速查卡

### DataFrame 基础（Day 13）

| 操作 | 代码 | 注意 |
|------|------|------|
| 读取 | `pd.read_csv('xxx.csv')` | 自动推断类型，日期需手动转 |
| 属性 | `df.shape / .columns / .dtypes / .info()` | info() 看非空计数 |
| 查看 | `df.head() / .tail() / .describe()` | describe() 只看数值列 |
| 单列 | `df['col']` → Series | |
| 多列 | `df[['a','b']]` → DataFrame | 双层列表 |
| loc | `df.loc[行标签, 列标签]` | 含头含尾 |
| iloc | `df.iloc[行位置, 列位置]` | 左闭右开 |
| 布尔筛选 | `df[df['x'] > 100]` | 多条件用 `&` + 括号 |
| isin | `df['col'].isin([...])` | 多值匹配 |
| 赋值 | `df.loc[条件, 'col'] = val` | 不用链式赋值 |
| 新增列 | `df['new'] = df['a'] / df['b']` | 向量化广播 |
| 排序 | `df.sort_values('col')` | 返回新对象，需赋值 |
| 去重 | `df.drop_duplicates()` | 按行去重 |
| 类型转换 | `pd.to_numeric(col, errors='coerce')` | 失败变 NaN |
| 日期 | `pd.to_datetime(col)` | `.dt.year/.dt.month` |

### 字符串/缺失值/分箱（Day 14）

| 操作 | 代码 | 注意 |
|------|------|------|
| 包含 | `df['col'].str.contains('x')` | 模糊匹配 |
| 替换 | `df['col'] = df['col'].str.replace('a','b')` | **必须赋值** |
| 长度 | `df['col'].str.len()` | 验证字段长度 |
| apply | `df.apply(func, axis=1)` | 慢，优先用向量化 |
| np.where | `np.where(df['x']>0, '高', '低')` | 向量化 if-else |
| 删除缺失 | `df.dropna()` | 返回新对象 |
| 填充缺失 | `df['col'] = df['col'].fillna(0)` | **必须赋值** |
| 分组填充 | `df.groupby('g')['x'].transform('mean')` | 等长返回 |
| 分箱(等宽) | `pd.cut(col, bins=[...])` | 按数值区间 |
| 分箱(等分位) | `pd.qcut(col, q=4)` | 每箱数量相等 |
| 透视表 | `pd.pivot_table(...)` | 行列统计汇总 |

### groupby/merge/concat（Day 15）

| 操作 | 代码 | 返回值 |
|------|------|--------|
| 分组聚合 | `df.groupby('g')['x'].sum()` | Series |
| 多聚合 | `df.groupby('g')['x'].agg(['sum','mean'])` | DataFrame |
| 命名聚合 | `df.groupby('g').agg(sum=('x','sum'))` | DataFrame |
| transform | `df.groupby('g')['x'].transform('mean')` | 等长Series |
| 分组排名 | `df.groupby('g')['x'].transform('rank')` | 等长Series |
| filter | `df.groupby('g').filter(lambda x: ...)` | DataFrame |
| merge | `pd.merge(a, b, on='k', how='left')` | DataFrame |
| concat | `pd.concat([a,b], axis=0, ignore_index=True)` | DataFrame |

**核心心法**: 字符串操作用 `.str`，缺失值先判断再处理，能向量化就不 apply，分组聚合用 `groupby`/`pivot_table`，表关联用 `merge`（先想 INNER vs LEFT）。

## 2. Week 3 弱点清单回顾

| # | 弱点 | 触发场景 | 防御策略 |
|---|------|----------|----------|
| 42 | 读题不仔细 | 把 category 看成 product | 做完后回头看题目要求的列/参数/值 |
| 46 | isin 统计用 `.sum()` | `len(series.isin())` 返回长度 | 布尔面具用 `.sum()` 计数 |
| 47 | 多条件必加括号 | `(a>0) & (b<10)` | 每个条件独立括起来 |
| 48 | 方法返回新对象 | `.str.replace` / `.fillna` 未赋值 | **必须显式赋值回 df** |
| 49 | sort_values 不修改原 df | `sort_values` 后 `loc[0:9]` 改错 | 用 `nlargest` 取索引再赋值 |
| 50 | 方法调用加括号 | `.describe` 没括号 | 方法必须 `()` 调用 |
| 51 | 日期分析确认范围 | today 选在订单日期前 | 先 `df['date'].max()` 确认范围 |
| 52 | 数值计算用数值类型 | RFM_Score 字符串拼接 | 用 `int()` 相加再排序 |

## 3. 综合场景演练 —— 数据分析师的完整工作流

**场景**: 你收到了一份门店销售数据 `sales.csv` 和客户数据 `customers.csv`，需要完成一次完整的数据分析项目。

**任务**: 从读取 → 清洗 → 分析 → 报告，完整走一遍数据岗工作流。

In [ ]:
import pandas as pd
import numpy as np
import json

# ========== 阶段1: 读取 ==========
df = pd.read_csv('../data/sales.csv')
customers = pd.read_csv('../data/customers.csv')

print(f"Sales: {df.shape}, Customers: {customers.shape}")

# ========== 阶段2: 清洗 ==========
# 日期转换
df['order_date'] = pd.to_datetime(df['order_date'], errors='coerce')

# 检查缺失值
print(df.isnull().sum())

# 人为制造几个缺失值来练习填充（实际项目中跳过）
df.loc[df.sample(5).index, 'total'] = np.nan

# 按国家分组均值填充
df['total_filled'] = df['total'].fillna(
    df.groupby('country')['total'].transform('mean')
)

# 单价列
df['unit_price'] = df['total_filled'] / df['quantity']

# 等级列
df['price_level'] = np.where(
    df['total_filled'] >= 5000, '高',
    np.where(df['total_filled'] >= 1000, '中', '低')
)

# ========== 阶段3: 关联 ==========
merged = pd.merge(df, customers, on='customer_id', how='left')
unmatched = merged[merged['name'].isnull()]
print(f"未匹配订单: {len(unmatched)} / {len(merged)} ({len(unmatched)/len(merged)*100:.1f}%)")

# ========== 阶段4: 分析 ==========
# 4.1 按国家分组统计
country_stats = merged.groupby('country').agg(
    order_count=('order_id', 'count'),
    total_sales=('total_filled', 'sum'),
    avg_order=('total_filled', 'mean'),
    max_order=('total_filled', 'max')
).reset_index()
print(country_stats)

# 4.2 透视表: 国家 × 品类
pivot = pd.pivot_table(
    merged, values='total_filled', index='country',
    columns='category', aggfunc='sum', fill_value=0
)
print(pivot)

# 4.3 客户排名（每个国家内）
merged['country_rank'] = merged.groupby('country')['total_filled'].transform(
    'rank', ascending=False, method='first'
)
top_per_country = merged[merged['country_rank'] == 1][
    ['country', 'name', 'total_filled']
]
print(top_per_country)

# ========== 阶段5: 输出报告 ==========
report = {
    'total_orders': len(merged),
    'total_sales': float(merged['total_filled'].sum()),
    'avg_order': float(merged['total_filled'].mean()),
    'top_country': country_stats.loc[country_stats['total_sales'].idxmax(), 'country'],
    'top_country_sales': float(country_stats['total_sales'].max()),
}

with open('week3_report.json', 'w', encoding='utf-8') as f:
    json.dump(report, f, ensure_ascii=False, indent=2)

print(json.dumps(report, ensure_ascii=False, indent=2))